In [ ]:
import numpy as np
from collections import Counter
from colorama import Fore, Style

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from lstm import William

In [2]:
# Data

def preprocess(text):
    text = text.strip().lower()
    text = text.replace("\n\n", "\n")
    for (redund, symb) in {
        "1": "0",
        "2": "0",
        "3": "0",
        "4": "0",
        "5": "0",
        "6": "0",
        "7": "0",
        "8": "0",
        "9": "0",
    }.items(): text = text.replace(redund, symb)
    return text

ALL_SHAKESPEARE = 'data/shakespeare.txt'
MIDSUMMER_NIGHT = 'data/a-midsummer-nights-dream.txt'

data = ""
with open(MIDSUMMER_NIGHT, 'r') as sh:
    for line in sh.readlines():
        data += line
data = preprocess(data)

In [23]:
# BPE

def encode_pairs(text, replacements):
    for (byte, pair) in replacements.items():
        text = text.replace(byte, pair)
    return text

text = data[335:]
tokenizer = {char: str(byte) for (byte, char) in enumerate(list(Counter(text).keys()))}
decoder = {str(byte): char for (char, byte) in tokenizer.items()}

byte_array = [tokenizer[c] for c in list(text)]
byte_pairs = [(byte_array[i], byte_array[i+1]) for i in range(len(byte_array)-1)]

log = {}

while Counter(byte_pairs).most_common()[0][1] >= 10:
    # Find pairs
    top_pair = Counter(byte_pairs).most_common()[0][0]
    top_bigram = decoder[top_pair[0]] + decoder[top_pair[1]]
    top_pair = " " + top_pair[0] + " " + top_pair[1] + " "

    # Updates
    log[top_pair] = " " + str(len(tokenizer)) + " "
    byte_array = (" " + " ".join(byte_array) + " ").replace(top_pair, " " + str(len(tokenizer)) + " ").strip().split(" ")
    byte_pairs = [(byte_array[i], byte_array[i+1]) for i in range(len(byte_array)-1)]
    tokenizer[top_bigram] = str(len(tokenizer))
    decoder[str(len(decoder))] = top_bigram

print(len(tokenizer))

962


In [26]:
# Sequences

sequences = []
targets = []
seq_length = 30
for x in range(len(byte_array) - seq_length):
    sequences += [[int(d) for d in byte_array[x:x+seq_length]]]
    targets += [int(byte_array[x+seq_length])]

X = torch.tensor(sequences, dtype=int)
y = torch.tensor(targets, dtype=int)
print(X.shape, y.shape)

torch.Size([32555, 30]) torch.Size([32555])


In [ ]:
# Training

def train_model(model, criterion, optimizer, X, n_epochs=35, batch_size=1000):
    h0, c0 = None, None
    increment = X.shape[0] // batch_size
    for epoch in range(n_epochs):
        epoch_loss = 0
        for i in range(increment):
            idx = i*batch_size
        
            model.train()
            optimizer.zero_grad()

            outputs, hn, cn = model(X[idx:idx+batch_size], (h0, c0))

            loss = criterion(outputs, y[idx:idx+batch_size])
            loss.backward()
            optimizer.step()

            # h0, c0 = hn.detach(), cn.detach()
            epoch_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}, Loss: {epoch_loss/increment:.4f}')
    print(f'Final Loss: {loss.item():.4f}')

input_dim = 175
hidden_dim = 175  # Use for cell and hidden state if proj_size = 0

bill = William(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    layer_dim=1,
    output_dim=len(tokenizer),
    dropout=0.15,
    bidirectional=2
)
criterion = nn.CrossEntropyLoss()
learning_rate = 0.003
optimizer = optim.Adam(bill.parameters(), lr=learning_rate)
n_epochs = 35
batch_size = 1000

train_model(bill, criterion, optimizer, X, n_epochs, batch_size)

In [67]:
# Download

MODEL_PATH = f"bill{n_epochs}-l{seq_length}-d{X.shape[0]}-Hi{input_dim}-Hc{hidden_dim}.pt"
torch.save(bill, MODEL_PATH)

array([0.5, 1. , 1.5])

In [74]:
# Generation

i = 30
seed_text = """Full of vexation come I, with complaint
Against my child, my daughter Hermia.--
Stand forth, Demetrius.--My noble lord,
This man hath my consent to marry her.--
Stand forth, Lysander.--And, my gracious duke,
This man hath bewitched the bosom of my child."""
seed_text = preprocess(seed_text)
current_seq = [int(c) for c in encode_pairs(" ".join([tokenizer[x] for x in seed_text]), log).split(" ")[:40]]
seed_text = "".join([decoder[str(t)] for t in current_seq])

generate_length = 60
temperatures = [0.75, 1.25]

# model = torch.load("bill50-d97970-e64-h150.pt", weights_only=False)
# model.eval()

bill.eval()

print(Style.BRIGHT + f"\033[4mSeed Text: Green; Generated Text: Red\033[0m")
for temperature in temperatures:
    # seed_text = "shall i compare thee to a summer's day?\n"
    generated_text = ""
    with torch.no_grad():
        for _ in range(generate_length):
            x_input = torch.tensor([current_seq])
            
            logits = bill(x_input)[0]
            probs = torch.softmax(logits/temperature, dim=1).squeeze()
            
            next_char_idx = torch.multinomial(probs, 1).item()      
            generated_text += decoder[str(next_char_idx)]
            
            # Slide the window (remove first char, append predicted char)
            current_seq = current_seq[1:] + [next_char_idx]

    print(f"Temperature {temperature}:")
    print(Style.RESET_ALL + Fore.GREEN + seed_text + Fore.RED + generated_text + Style.RESET_ALL + "\n")

Seed Text: Green; Generated Text: Red
Temperature 0.75:
full of vexation come i, with complaint
against my child, my daughter hermia.--
stand forth, demetrius.--my noble lord,
this man hath my consent to marry her.--
stand forth, lysander.--and, my gracious duke,
this man hath bewitched the bosoms inclout
to seek forbush this i thy faith thorough be 

Temperature 1.25:
full of vexation come i, with complaint
against my child, my daughter hermia.--
stand forth, demetrius.--my noble lord,
this man those that is my lord, when the more.
i plowwormurt it stands hold me their slameassixouson this ports,
	   sightylt,
she man mlove stret
from the  himest b that f

